In [1]:
from paths import *
import os
import numpy as np
from scipy.stats import ttest_rel, wilcoxon

In [14]:
# Configs
envS_path = os.path.join(model_save_, f"eval-C_0Tj-1021123825")
envP_path = os.path.join(model_save_, f"eval-C_0Tk-1021183234")
task_name = "ABXSomething-stop-data-aspirationRangeComp"
range_start, range_end = 2, 4
model_condition = "b"
strseq_learned_runs = "12345"


In [15]:
# Collect data
envS_res = {}
envP_res = {}
for hiddim in [3, 8, 16, 32, 48, 64]: 
    model_type = f"recon{hiddim}-phi"
    for environment, env_res, res_save_dir in zip(["S", "P"], [envS_res, envP_res], [envS_path, envP_path]):
        layered_res = {}
        look_for_layer_path = f"{task_name}-{range_start}-{range_end}"
        # Read ori
        ori_path = os.path.join(res_save_dir, look_for_layer_path, f"07-save-ari-recon64-phi-{model_condition}-{strseq_learned_runs}-ori.npy")
        if os.path.exists(ori_path): 
            ori_res = np.load(ori_path)
        else: 
            raise ValueError("No ori path found.")
        
        for layer in ["hidrep", "attnout", "dec-lin1", "enc-lin1", 
                        "dec-rnn1-f", "enc-rnn1-f", "enc-rnn1-b",
                        "dec-rnn2-f", "enc-rnn2-f", "enc-rnn2-b", 
                        "dec-rnn3-f", "enc-rnn3-f", "enc-rnn3-b", 
                        "dec-rnn4-f", "enc-rnn4-f", "enc-rnn4-b", 
                        "dec-rnn5-f", "enc-rnn5-f", "enc-rnn5-b", ]: # "enc-lin1", 
            print(f"Processing {model_type} in layer {layer}...")
            asp_list_epochs = []
            layer_path = os.path.join(res_save_dir, look_for_layer_path, 
                                        f"07-save-ari-{model_type}-{model_condition}-{strseq_learned_runs}-{layer}.npy")
            if os.path.exists(layer_path): 
                layer_res = np.load(layer_path)
            else: 
                print(f"Warning: {layer_path} not found. ")
                layer_res = np.zeros_like(ori_res)
            layered_res[layer] = layer_res
        layered_res["ori"] = ori_res
        env_res[hiddim] = layered_res


Processing recon3-phi in layer hidrep...
Processing recon3-phi in layer attnout...
Processing recon3-phi in layer dec-lin1...
Processing recon3-phi in layer enc-lin1...
Processing recon3-phi in layer dec-rnn1-f...
Processing recon3-phi in layer enc-rnn1-f...
Processing recon3-phi in layer enc-rnn1-b...
Processing recon3-phi in layer dec-rnn2-f...
Processing recon3-phi in layer enc-rnn2-f...
Processing recon3-phi in layer enc-rnn2-b...
Processing recon3-phi in layer dec-rnn3-f...
Processing recon3-phi in layer enc-rnn3-f...
Processing recon3-phi in layer enc-rnn3-b...
Processing recon3-phi in layer dec-rnn4-f...
Processing recon3-phi in layer enc-rnn4-f...
Processing recon3-phi in layer enc-rnn4-b...
Processing recon3-phi in layer dec-rnn5-f...
Processing recon3-phi in layer enc-rnn5-f...
Processing recon3-phi in layer enc-rnn5-b...
Processing recon3-phi in layer hidrep...
Processing recon3-phi in layer attnout...
Processing recon3-phi in layer dec-lin1...
Processing recon3-phi in layer

In [46]:
# Define test functions
def test_arrays(data1, data2, test_epoch_range=(0, 101)): 
    # Assuming `data1` and `data2` are your two numpy arrays of shape (num_runs, num_epochs)
    # Choose your test (paired t-test or Wilcoxon signed-rank test)
    p_values = {}
    t_stats = {}

    # Loop over each epoch to perform the significance test
    for epoch in test_epoch_range:
        t_stats[epoch], p_values[epoch] = ttest_rel(data1[:, epoch], data2[:, epoch])
    return t_stats, p_values

def test_arrays_merged(data1, data2, test_epoch_range=(0, 101)): 
    # Assuming `data1` and `data2` are your two numpy arrays of shape (num_runs, num_epochs)
    # Choose your test (paired t-test or Wilcoxon signed-rank test)

    data1_pooled = data1[:, test_epoch_range[0]:test_epoch_range[1]].flatten()
    data2_pooled = data2[:, test_epoch_range[0]:test_epoch_range[1]].flatten()

    return ttest_rel(data1_pooled, data2_pooled)

In [50]:
# Shape of each layer's result: (num_runs, num_epochs)
test_hiddim = 32
test_epoch_range = range(50, 101)
envS_ori_attn_diff = envS_res[test_hiddim]["attnout"] - envS_res[test_hiddim]["ori"]
envP_ori_attn_diff = envP_res[test_hiddim]["attnout"] - envP_res[test_hiddim]["ori"]

envS_ori_attn_diff_mag = envS_ori_attn_diff * 1
envP_ori_attn_diff_mag = envP_ori_attn_diff * 1
# Shape: (num_runs, num_epochs)

t_stats, p_values = test_arrays(envS_ori_attn_diff_mag, envP_ori_attn_diff_mag, test_epoch_range)
merged_t_stat, merged_p_value = test_arrays_merged(envS_ori_attn_diff_mag, envP_ori_attn_diff_mag, test_epoch_range)


In [51]:
merged_t_stat, merged_p_value

(-1.6857459873600684, 0.10257684464151981)

In [52]:
envS_ori_attn_diff_mag[:, 95]

array([-0.00619048, -0.02444444,  0.00746032,  0.02222222, -0.06412698,
       -0.07571429, -0.03666667, -0.06269841,  0.07047619, -0.05920635,
       -0.00444444, -0.05063492,  0.08761905,  0.07730159, -0.00650794,
        0.04253968,  0.04666667,  0.03714286, -0.04333333,  0.02825397,
       -0.00920635, -0.0415873 , -0.17857143, -0.16746032, -0.00539683,
       -0.04587302,  0.08015873, -0.06269841, -0.00349206, -0.0847619 ])

In [53]:
t_stats, p_values

({50: -1.6857459873600684,
  51: -3.0562129272211935,
  52: -1.7593288296054659,
  53: -1.7357076558024,
  54: 0.3073847997250918,
  55: -2.235306110419264,
  56: -1.4644643947612308,
  57: -3.7110550320671583,
  58: -0.49064259258871284,
  59: -0.6181297725879507,
  60: -0.5724704807308001,
  61: -3.1383887950213243,
  62: -2.4049456956896305,
  63: -1.827912775318576,
  64: -0.475889120726835,
  65: -0.8321390734906149,
  66: -1.6268581730096228,
  67: -1.9895083262515472,
  68: 0.021071771585940504,
  69: -1.699985794284464,
  70: -1.9368765851441392,
  71: -0.8593258599789243,
  72: -1.699235378431617,
  73: -0.6353800033050948,
  74: -2.042780350973025,
  75: 0.14939918284481912,
  76: -0.969821753514799,
  77: -0.1747847817074975,
  78: -1.2689573868029256,
  79: -1.2145205670113848,
  80: 1.1471113381559903,
  81: -1.7255790740740002,
  82: -0.9941081236624354,
  83: -0.8653274017746192,
  84: -0.6625569432183119,
  85: -2.858777965555683,
  86: -1.9798569044358416,
  87: -1.257

In [24]:
from statsmodels.stats.power import TTestPower

# Parameters
sample_size = 30      # Sample size per group (assuming paired samples, n = 30 in total)
alpha = 0.05          # Significance threshold
power = 0.8           # Desired power (often 0.8)

# Initialize power analysis object for a paired t-test
power_analysis = TTestPower()

# Calculate effect size
effect_size = power_analysis.solve_power(nobs=sample_size, alpha=alpha, power=power, alternative='two-sided')
print("Minimum detectable effect size:", effect_size)


Minimum detectable effect size: 0.5292357068515448
